# Code of the methods

In [ ]:
import math
from typing import Tuple

import torch
import torch.nn.functional as F
from torch import Tensor
from typing import List, Callable
import numpy as np
from scipy.spatial import cKDTree
from torch.func import vjp, vmap

def compute_K(x: Tensor, eps: float = 1e-6) -> Tensor:
    x_norm = (x ** 2).sum(dim=1, keepdim=True)
    K = x_norm + x_norm.T - 2 * x @ x.T
    upper_tri_mask = torch.triu(torch.ones_like(K, dtype=torch.bool), diagonal=1)
    K_upper = K[upper_tri_mask]
    median_val = torch.median(K_upper).detach() + eps
    return K / median_val, K_upper / median_val

def logdet_psd_cholesky(K: torch.Tensor, jitter: float = 1e-3) -> torch.Tensor:
    B = K.size(0)
    I = torch.eye(B, dtype=K.dtype, device=K.device)
    L, info = torch.linalg.cholesky_ex(K + jitter * I)
    logdet = 2.0 * torch.log(torch.diagonal(L, dim1=-2, dim2=-1)).sum(dim=-1)
    return logdet

class FlowToGMM(torch.nn.Module):
    def __init__(self, mus: Tensor, sigmas: Tensor, weights: Tensor):
        super().__init__()
        assert mus.ndim == 2 and sigmas.shape == mus.shape
        assert weights.ndim == 1 and weights.size(0) == mus.size(0)
        assert (sigmas > 0).all()
        self.register_buffer("mus", mus)
        self.register_buffer("sigmas", sigmas)
        self.register_buffer("weights", weights / weights.sum())
        self.register_buffer("base_sigma", torch.tensor(1.0))

    def _params_at(self, t: Tensor) -> Tuple[Tensor, Tensor]:
        t = t.to(dtype=self.mus.dtype, device=self.mus.device)
        m_t = t * self.mus
        Sigma_t = (1.0 - t) ** 2 * self.base_sigma + (t ** 2) * self.sigmas
        Sigma_t = Sigma_t.clamp_min(1e-12)
        return m_t, Sigma_t

    def _responsibilities(self, x: Tensor, t: Tensor):
        m_t, Sigma_t = self._params_at(t)
        dx = x[:, None, :] - m_t[None, :, :]
        log_prob = -0.5 * (
            (dx ** 2 / Sigma_t[None]).sum(-1) +
            torch.log(2 * torch.pi * Sigma_t[None]).sum(-1)
        )
        gamma = torch.softmax(torch.log(self.weights)[None, :] + log_prob, dim=1)
        return gamma, m_t, Sigma_t

    def velocity(self, x: Tensor, t: float) -> Tensor:
        t = torch.tensor(t, dtype=self.mus.dtype, device=self.mus.device)
        gamma, m_t, Sigma_t = self._responsibilities(x, t)
        A_diag = (t * self.sigmas - (1.0 - t) * self.base_sigma) / Sigma_t
        xm = x[:, None, :] - m_t[None, :, :]
        v_k = A_diag[None] * xm + self.mus[None]
        return (gamma[:, :, None] * v_k).sum(1)

    def velocity_batched(self, x: Tensor, t: Tensor) -> Tensor:
        device, dtype = self.mus.device, self.mus.dtype

        t = torch.as_tensor(t, device=device, dtype=dtype)
        assert t.ndim == 2 and t.shape[1] == 1

        one_m_t = (1.0 - t).clamp_min(0.0)

        m_t = t[:, None, :] * self.mus[None, :, :]
        Sigma_t = (one_m_t**2)[:, None, :] * self.base_sigma + (t**2)[:, None, :] * self.sigmas[None, :, :]
        Sigma_t = Sigma_t.clamp_min(1e-12)

        dx = x[:, None, :] - m_t
        log_prob = -0.5 * ((dx*dx / Sigma_t).sum(-1) + torch.log(2*torch.pi*Sigma_t).sum(-1))
        gamma = torch.softmax(torch.log(self.weights.clamp_min(1e-32))[None, :].to(device=device, dtype=dtype) + log_prob, dim=1)

        A_diag = (t[:, None, :] * self.sigmas[None, :, :] - one_m_t[:, None, :] * self.base_sigma) / Sigma_t
        v_k = A_diag * dx + self.mus[None, :, :]
        return (gamma[:, :, None] * v_k).sum(1)

    def dpp_diversity(self, x_req: Tensor, jitter: float = 1e-3) -> Tensor:
        K, _ = compute_K(x_req)
        L = torch.exp(-K)
        B = x_req.shape[0]
        I = torch.eye(B, dtype=L.dtype, device=L.device)
        logdet_num = logdet_psd_cholesky(L, jitter=jitter)
        logdet_den = logdet_psd_cholesky(L + I, jitter=jitter)
        return logdet_num - logdet_den

    def dpp_ho_diversity(self, x_req: torch.Tensor,
                            R: int = None,
                            use_ratio: bool = True,
                            jitter: float = 1e-6) -> torch.Tensor:
        B = x_req.shape[0]
        assert B >= 2, "need at least two points"

        xhat = F.normalize(x_req, dim=1)
        L = xhat @ xhat.t()
        L = 0.5 * (L + L.t())

        if R is None:
            R = max(1, B - 1)

        device, dtype = L.device, L.dtype
        a = torch.empty(R + 1, dtype=dtype, device=device)
        a[0] = 1.0
        if R >= 1:
            r = torch.arange(1, R + 1, device=device, dtype=dtype)
            a[1:] = 2.0 * (1.0 - r / (R + 1.0))

        T0 = torch.ones_like(L)
        K = a[0] * T0
        if R >= 1:
            T1 = L
            K = K + a[1] * T1
        Tm2, Tm1 = T0, (T1 if R >= 1 else T0)
        for r in range(2, R + 1):
            Tr = 2.0 * L * Tm1 - Tm2
            K = K + a[r] * Tr
            Tm2, Tm1 = Tm1, Tr

        K = K / (a.sum() + 1e-12)
        K = 0.5 * (K + K.t())

        I = torch.eye(B, dtype=dtype, device=device)
        sign_num, logdet_num = torch.linalg.slogdet(K + jitter * I)
        if use_ratio:
            sign_den, logdet_den = torch.linalg.slogdet(K + I)
            diversity = logdet_num - logdet_den
        else:
            diversity = logdet_num
        return diversity

    @torch.no_grad()
    def iid_sample(self, n_samples: int, steps: int = 100, seed: int = None) -> Tensor:
        if seed is not None:
            torch.manual_seed(seed)
        x = torch.randn(n_samples, self.mus.shape[1], device=self.mus.device, dtype=self.mus.dtype) * torch.sqrt(self.base_sigma)
        t, dt = 0.0, 1.0 / steps
        for _ in range(steps):
            v = self.velocity(x, t)
            x = x + v * dt
            t += dt
        return {'samples': x.clone().detach()}

    def get_force(self, force_name, x, v, t, f_0, s_proj='none'):
        fmap = {
            'dpp': self.dpp_diversity,
            'dpp_ho': self.dpp_ho_diversity,
        }

        if force_name not in fmap:
            raise ValueError(f"Invalid force name: {force_name}")

        with torch.enable_grad():
            x_req = x.detach().requires_grad_(True)
            if x_req.shape[0] < 2:
                return torch.zeros_like(x_req)
            x_1 = x_req + self.velocity(x, t).detach() * (1 - t)
            diversity = fmap[force_name](x_1)
            (grad_x,) = torch.autograd.grad(diversity, x_req)
        f = grad_x.detach()

        score = self.get_score(x, t)
        score_normalized_per_sample = score / score.norm(dim=-1, keepdim=True)
        if s_proj == 'hard':
            f_parallel_scalar_per_sample = (f * score_normalized_per_sample).sum(-1, keepdim=True)
            f_parallel = f_parallel_scalar_per_sample * score_normalized_per_sample
            f_vertical = f - f_parallel
            f = f_vertical + F.relu(f_parallel_scalar_per_sample) * score_normalized_per_sample
        elif s_proj == 'soft':
            f_parallel_scalar_per_sample = (f * score_normalized_per_sample).sum(-1, keepdim=True)
            f_parallel = f_parallel_scalar_per_sample * score_normalized_per_sample
            f_vertical = f - f_parallel
            soft_factor = 1.0 - t
            f_parallel_scalar_soft = f_parallel_scalar_per_sample * soft_factor + F.relu(f_parallel_scalar_per_sample) * (1.0 - soft_factor)
            f = f_vertical + f_parallel_scalar_soft * score_normalized_per_sample
        elif s_proj == 'none':
            pass
        else:
            raise ValueError(f"Invalid s_proj: {s_proj}")

        f = f / (f.view(-1).norm() + 1e-3)
        f = f * v.view(-1).norm()

        f = f * math.sqrt(max(1.0 - t, 0.0) + 1e-6)

        f_final = f * f_0
        return f_final

    @torch.no_grad()
    def non_iid_sample(self, force_name, n_samples: int, f_0: float = 1.0, steps: int = 100, seed: int = None, save_path: bool = False, \
        s_proj='none') -> Tensor:
        if seed is not None:
            torch.manual_seed(seed)
        path_history, velocity_history, force_history, time_history = [], [], [], []
        x = torch.randn(n_samples, self.mus.shape[1], device=self.mus.device, dtype=self.mus.dtype) * torch.sqrt(self.base_sigma)
        t, dt = 0.0, 1.0 / steps
        for _ in range(steps):
            v = self.velocity(x, t)
            f = self.get_force(force_name, x, v, t, f_0, s_proj=s_proj)

            assert torch.isfinite(v).all(), "velocity has non-finite values"
            assert torch.isfinite(f).all(), "force has non-finite values"
            assert torch.isfinite(x).all(), "state x has non-finite values"

            if save_path:
                path_history.append(x.clone().detach())
                velocity_history.append(v.clone().detach())
                force_history.append(f.clone().detach())
                time_history.append(t)

            x = x + (v + f) * dt
            t += dt

        return {
            'samples': x.clone().detach(),
            'path_history': path_history,
            'velocity_history': velocity_history,
            'force_history': force_history,
            'time_history': time_history
        }

    def get_score(self, x: Tensor, t: float) -> Tensor:
        v = self.velocity(x, t)
        s = (t * v - x) / (1 - t)
        return s

    def compute_iid_log_density(self, x: Tensor) -> Tensor:
        x = x.to(dtype=self.mus.dtype, device=self.mus.device)
        B, D = x.shape

        mus = self.mus
        sig = self.sigmas.clamp_min(1e-12)
        w = self.weights.clamp_min(1e-32)

        dx = x[:, None, :] - mus[None, :, :]
        quad = (dx * dx / sig[None, :, :]).sum(dim=-1)
        log_det = torch.log(sig[None, :, :]).sum(dim=-1)
        log_comp = -0.5 * (quad + log_det + D * math.log(2.0 * math.pi))

        logw = torch.log(w)[None, :]
        log_p = torch.logsumexp(logw + log_comp, dim=1)

        return log_p

    def compute_non_iid_log_density_knn(self, x: torch.Tensor) -> torch.Tensor:
        ks_base = (10, 15, 25, 35, 50)
        neighbor_factor = 3
        shrink_rel = 1e-6
        shrink_abs = 1e-12
        target_bytes = 512 * 1024 * 1024
        unit_info = 1.0

        target_dtype = self.mus.dtype
        target_device = self.mus.device
        if x.dtype != target_dtype or x.device != target_device:
            x = x.to(dtype=target_dtype, device=target_device)
        B, D = x.shape

        if B <= 1:
            return torch.stack([
                torch.full((B,), float("-inf"), dtype=x.dtype, device=x.device),
                torch.zeros(B, dtype=x.dtype, device=x.device)
            ], dim=1)

        ks = [k for k in ks_base if k <= B - 1]
        assert len(ks) > 0, f"B={B} too small for ks_base={ks_base}"
        kmax = max(ks)

        n_total = float(B - 1)
        R_K = (4.0 * math.pi) ** (-0.5 * D)
        log_R_K = math.log(R_K)

        x_cpu = x.detach().to("cpu")
        X_np = x_cpu.to(torch.float64).numpy()
        tree = cKDTree(X_np, leafsize=64, balanced_tree=True, compact_nodes=True)

        Kbig = min(B - 1, max(kmax * neighbor_factor, kmax + 8))

        bytes_per_row = 16 * (Kbig + 1)
        qbatch = max(1, min(B, target_bytes // max(bytes_per_row, 1)))

        out_log = torch.empty(B, dtype=x.dtype, device=x.device)
        out_conf = torch.empty(B, dtype=x.dtype, device=x.device)

        tiny = np.finfo(np.float64).tiny

        for i in range(0, B, qbatch):
            i_end = min(i + qbatch, B)
            Xi = X_np[i:i_end]

            dists, idxs = tree.query(Xi, k=Kbig + 1, p=2, workers=-1)

            gidx = np.arange(i, i_end)
            R = i_end - i

            if np.all(idxs[:, 0] == gidx):
                idx_no = idxs[:, 1:Kbig + 1].copy()
                d_no = dists[:, 1:Kbig + 1].copy()
            else:
                idx_no = np.empty((R, Kbig), dtype=idxs.dtype)
                d_no = np.empty((R, Kbig), dtype=dists.dtype)
                for r in range(R):
                    idr = idxs[r]
                    ddr = dists[r]
                    pos = np.nonzero(idr == gidx[r])[0]
                    if pos.size:
                        sp = int(pos[0])
                        idr = np.concatenate((idr[:sp], idr[sp + 1:]))
                        ddr = np.concatenate((ddr[:sp], ddr[sp + 1:]))
                    idx_no[r] = idr[:Kbig]
                    d_no[r] = ddr[:Kbig]

            a_h_batch = np.full((R, len(ks)), -np.inf, dtype=np.float64)
            logI_h_batch = np.full((R, len(ks)), -np.inf, dtype=np.float64)

            for r in range(R):
                center = Xi[r]
                nbr_idx = idx_no[r]
                nbr_xy  = X_np[nbr_idx]
                U = nbr_xy - center
                r_sorted = d_no[r]

                for j, k in enumerate(ks):
                    h = max(r_sorted[k - 1], 1e-12)

                    Kuse = min(Kbig, max(k * neighbor_factor, k + 8))
                    U_k = U[:Kuse]

                    q = np.sum(U_k * U_k, axis=1) / (2.0 * h * h)
                    w = np.exp(-q) / (((2.0 * math.pi) ** (0.5 * D)) * (h ** D))

                    sum_w = float(np.sum(w))
                    if (not np.isfinite(sum_w)) or (sum_w <= tiny):
                        continue

                    m0 = sum_w
                    m1 = (w[:, None] * U_k).sum(axis=0)
                    M2 = (w[:, None, None] * (U_k[:, :, None] * U_k[:, None, :])).sum(axis=0)

                    mu = m1 / m0
                    S = M2 / m0 - np.outer(mu, mu)

                    S = 0.5 * (S + S.T)
                    trS = float(np.trace(S))
                    lam = max(shrink_abs, shrink_rel * (trS / D if trS > 0 else 1.0))
                    ev = np.linalg.eigvalsh(S)
                    if ev.min() < lam:
                        S += (lam - ev.min() + 1e-15) * np.eye(D)

                    L = np.linalg.cholesky(S)
                    logdetS = 2.0 * np.log(np.diag(L)).sum()
                    y = np.linalg.solve(L, mu)
                    mu_A_mu = float(y @ y)

                    a_hat = math.log(m0 / n_total) + D * math.log(h) - 0.5 * logdetS - 0.5 * mu_A_mu

                    sum_w2 = float((w * w).sum())
                    if sum_w2 <= tiny:
                        continue
                    n_eff = (m0 * m0) / sum_w2

                    logI_h = math.log(n_eff) + D * math.log(h) + a_hat - log_R_K
                    a_h_batch[r, j] = a_hat
                    logI_h_batch[r, j] = logI_h

            a_ivw = np.full(R, -np.inf, dtype=np.float64)
            conf = np.zeros(R, dtype=np.float64)
            log_unit_info = math.log(unit_info)
            for r2 in range(R):
                logI = logI_h_batch[r2]
                a_row = a_h_batch[r2]
                mask = np.isfinite(logI) & np.isfinite(a_row)
                if not np.any(mask):
                    continue
                mlog = float(np.max(logI[mask]))
                w = np.exp(logI[mask] - mlog)
                sumw = float(np.sum(w))
                if sumw <= tiny:
                    continue
                a_ivw[r2] = float(np.dot(w, a_row[mask]) / sumw)
                logI_sum = mlog + math.log(sumw)
                conf[r2] = 1.0 / (1.0 + float(np.exp(log_unit_info - logI_sum)))
            conf = np.clip(conf, 0.0, 1.0)

            out_log[i:i_end] = torch.from_numpy(a_ivw).to(device=x.device, dtype=x.dtype)
            out_conf[i:i_end] = torch.from_numpy(conf).to(device=x.device, dtype=x.dtype)

        return torch.stack([out_log, out_conf], dim=1)

    def importance_weight_computation(
        self,
        x_list: List[Tensor],
        force_list: List[Tensor],
        t_list: List[float],
        dvnet: Callable[[Tensor, Tensor], Tensor],
    ) -> Tensor:
        dvnet.eval()

        assert len(x_list) == len(force_list) == len(t_list), "Lists must be same length."
        B, D = x_list[0].shape
        device, dtype = x_list[0].device, x_list[0].dtype

        logw    = torch.zeros(B, device=device, dtype=dtype)

        def _divergence_per_sample_vjp(u_fn, x: Tensor) -> Tensor:
            B_, D_ = x.shape
            div = x.new_zeros(B_)
            eyeB = torch.eye(B_, device=x.device, dtype=x.dtype)
            for d in range(D_):
                def F_d(z):
                    return u_fn(z)[:, d]
                _, vjp_fn = vjp(F_d, x)
                grads = vmap(lambda e: vjp_fn(e)[0])(eyeB)
                div = div + grads[torch.arange(B_, device=x.device),
                                torch.arange(B_, device=x.device), d]
            return div

        for step_idx, (x_t, hat_g_t, t) in enumerate(zip(x_list, force_list, t_list)):
            x_req = x_t.detach().requires_grad_(True)

            assert isinstance(t, float)
            with torch.no_grad():
                next_t = 1.0 if (step_idx == len(t_list) - 1) else t_list[step_idx + 1]
                dt = (next_t - t)
                t_tensor = torch.ones(B, 1, device=device, dtype=dtype) * t

            delta_v_fn = lambda z: dvnet(z, t_tensor)
            div_delta_v = _divergence_per_sample_vjp(delta_v_fn, x_req)

            with torch.no_grad():
                delta_v_val = delta_v_fn(x_req)
                delta_s = delta_v_val * t / (1.0 - t)
                assert isinstance(t, float)
                v_marginal = dvnet(x_req, t_tensor) + self.velocity(x_req, t)
                s_non_iid = (t * v_marginal - x_req) / (1.0 - t)

            dlogw_dt = div_delta_v + (s_non_iid * delta_v_val).sum(dim=-1) - (delta_s * hat_g_t).sum(dim=-1)

            logw    = logw   + dt * dlogw_dt

        return logw

    def importance_weight_computation_fixed(
        self,
        samples: Tensor,
        t_list: List[float],
        dvnet: Callable[[Tensor, Tensor], Tensor],
    ) -> Tensor:
        dvnet.eval()

        B, D = samples.shape
        device, dtype = samples.device, samples.dtype

        logw    = torch.zeros(B, device=device, dtype=dtype)

        def _divergence_per_sample_vjp(u_fn, x: Tensor) -> Tensor:
            B_, D_ = x.shape
            div = x.new_zeros(B_)
            eyeB = torch.eye(B_, device=x.device, dtype=x.dtype)
            for d in range(D_):
                def F_d(z):
                    return u_fn(z)[:, d]
                _, vjp_fn = vjp(F_d, x)
                grads = vmap(lambda e: vjp_fn(e)[0])(eyeB)
                div = div + grads[torch.arange(B_, device=x.device),
                                torch.arange(B_, device=x.device), d]
            return div

        for step_idx, t in enumerate(t_list):
            x_req = samples.detach().requires_grad_(True)

            assert isinstance(t, float)
            with torch.no_grad():
                next_t = 1.0 if (step_idx == len(t_list) - 1) else t_list[step_idx + 1]
                dt = (next_t - t)
                t_tensor = torch.ones(B, 1, device=device, dtype=dtype) * t

            delta_v_fn = lambda z: dvnet(z, t_tensor)
            div_delta_v = _divergence_per_sample_vjp(delta_v_fn, x_req)

            with torch.no_grad():
                delta_v_val = delta_v_fn(x_req)
                delta_s = delta_v_val * t / (1.0 - t)
                assert isinstance(t, float)
                v_iid = self.velocity(x_req, t)
                v_marginal = dvnet(x_req, t_tensor) + v_iid
                s_non_iid = (t * v_marginal - x_req) / (1.0 - t)

            dlogw_dt = div_delta_v + (s_non_iid * delta_v_val).sum(dim=-1) + (delta_s * v_iid).sum(dim=-1)

            logw    = logw   + dt * dlogw_dt

        return logw

# Code for diverse sampling

In [ ]:
import torch
import matplotlib.pyplot as plt
from tqdm import tqdm

torch.manual_seed(0)

N = 10

all_dimensions = 8
radius = 1.0
sigma_XY = 0.01
sigma_others = 0.0001
angles = torch.linspace(0, 2 * torch.pi, N + 1)[:-1]
mus_torch = torch.zeros((N, all_dimensions))
mus_torch[:, :2] = torch.stack([radius * torch.cos(angles), radius * torch.sin(angles)], dim=1)
sigmas_torch = sigma_XY * torch.ones((N, all_dimensions))
sigmas_torch[:, 2:] = sigma_others

weights_torch = torch.tensor([1 for i in range(N)], dtype=torch.float)
weights_torch /= weights_torch.sum()

from typing import Sequence

def expected_unique(weights: Sequence[float], n: int = 10) -> float:
    total = float(sum(weights))
    if total <= 0:
        raise ValueError("Sum of weights must be positive.")
    ps = [w / total for w in weights]
    return sum(1.0 - (1.0 - p) ** n for p in ps)
print('expected coverage:', expected_unique(weights_torch.tolist(), n=N) / N)

def sample_gmm_torch(n_samples):
    samples = []
    component_choices = torch.multinomial(weights_torch, n_samples, replacement=True)
    for k in component_choices:
        mean = mus_torch[k]
        std = torch.sqrt(sigmas_torch[k])
        sample = mean + std * torch.randn(all_dimensions)
        samples.append(sample)
    return torch.stack(samples)

gmm_samples_torch = sample_gmm_torch(10000)

plt.figure(figsize=(6,6))
plt.scatter(gmm_samples_torch[:,0].numpy(), gmm_samples_torch[:,1].numpy(), alpha=0.1, s=10, label="Samples")
plt.scatter(mus_torch[:,0].numpy(), mus_torch[:,1].numpy(), color='red', marker='x', s=100, label="Component Means")
plt.title('Samples from 2D Gaussian Mixture Model')
plt.xlabel('x')
plt.ylabel('y')
plt.axis('equal')
plt.grid(True)
plt.legend()
plt.show()

flow = FlowToGMM(mus_torch, sigmas_torch, weights_torch)

In [ ]:
output = flow.non_iid_sample('dpp', 10, steps=100, save_path=True, f_0=1.0, s_proj='hard')
samples = output['samples']
path_history = output['path_history']
velocity_history = output['velocity_history']
force_history = output['force_history']

import matplotlib.pyplot as plt

arrow_length = 0.2

plt.figure(figsize=(10, 10))

for i in range(len(samples)):
    path = torch.stack([p[i] for p in path_history])
    plt.plot(path[:, 0].cpu(), path[:, 1].cpu(), alpha=0.3, linewidth=0.5)

subsample = 5
for step_idx in range(0, len(path_history) - 1, subsample):
    for i in range(len(samples)):
        pos = path_history[step_idx][i].cpu()

        if step_idx < len(velocity_history):
            v = velocity_history[step_idx][i].cpu()
            v_norm = v / (torch.norm(v) + 1e-8) * 0.3
            plt.arrow(pos[0], pos[1], v_norm[0] * arrow_length, v_norm[1] * arrow_length,
                     head_width=0.02, head_length=0.02, fc='blue', ec='blue', alpha=0.4)

        if step_idx < len(force_history):
            f = force_history[step_idx][i].cpu()
            f_norm = f / (torch.norm(f) + 1e-8) * 0.3
            plt.arrow(pos[0], pos[1], f_norm[0] * arrow_length, f_norm[1] * arrow_length,
                     head_width=0.02, head_length=0.02, fc='red', ec='red', alpha=0.4)

plt.scatter(samples[:, 0].cpu(), samples[:, 1].cpu(), s=50, c='green', marker='o', label='Final samples', zorder=5)
plt.scatter(mus_torch[:, 0].cpu(), mus_torch[:, 1].cpu(), s=100, c='black', marker='x', label='Means', zorder=5)

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color='green', marker='o', linestyle='', markersize=8, label='Final samples'),
    Line2D([0], [0], color='black', marker='x', linestyle='', markersize=8, label='Means'),
    Line2D([0], [0], color='blue', marker='>', linestyle='', markersize=8, label='Velocity direction'),
    Line2D([0], [0], color='red', marker='>', linestyle='', markersize=8, label='Force direction'),
    Line2D([0], [0], color='gray', linestyle='-', linewidth=1, label='Sample paths')
]

plt.axis('equal')
plt.grid(True, alpha=0.3)
plt.legend(handles=legend_elements)
plt.title('Flow paths with velocity and force directions')
plt.show()

In [ ]:
import torch
import numpy as np
import random

from scipy import stats
def summarize_with_ci(values, confidence=0.95, round_to=6):
    arr = np.array(values)
    mean = np.mean(arr)
    std = np.std(arr, ddof=1)
    n = len(arr)
    se = std / np.sqrt(n)
    t_val = stats.t.ppf((1 + confidence) / 2, df=n - 1)
    ci_half = t_val * se

    mean = round(float(mean), round_to)
    std = round(float(std), round_to)
    ci_half = round(float(ci_half), round_to)
    return {
        "mean": mean,
        "std": std,
        "ci_half": ci_half
    }

def evaluate_mode_coverage(
    flow, force_name, mode_means, num_samples=10, repeat_time=100, s_proj='hard', f_0=1.0, base_seed=0.0
):
    device = mode_means.device
    num_modes = mode_means.size(0)

    M = mode_means.to(device)
    M_sq = (M ** 2).sum(dim=1)

    def _single_run(run_idx: int) -> int:
        torch.manual_seed(base_seed + run_idx)
        torch.cuda.manual_seed(base_seed + run_idx)
        random.seed(base_seed + run_idx)
        np.random.seed(base_seed + run_idx)

        out = flow.non_iid_sample(
            force_name, num_samples,
            s_proj=s_proj,
            f_0=f_0
        )

        X = out["samples"].detach().to(device)
        X_sq = (X ** 2).sum(dim=1, keepdim=True)
        dist2 = X_sq + M_sq.unsqueeze(0) - 2 * (X @ M.t())

        min_dist2, argmin = dist2.min(dim=1)

        log_p = flow.compute_iid_log_density(X)
        return torch.unique(argmin).numel(), (min_dist2.mean()).sqrt().item(), log_p.mean().item(), X

    all_coverage_rate = []
    all_rmse = []
    all_log_p = []
    all_X = []
    for i in range(repeat_time):
        coverage_count, rmse, log_p, X = _single_run(i)
        all_coverage_rate.append(coverage_count / num_modes)
        all_rmse.append(rmse)
        all_log_p.append(log_p)
        all_X.append(X)

    coverage_summary = summarize_with_ci(all_coverage_rate)
    rmse_summary = summarize_with_ci(all_rmse)
    log_p_summary = summarize_with_ci(all_log_p)

    all_X = torch.cat(all_X, dim=0)
    ind = torch.randperm(all_X.size(0))
    all_X = all_X[ind]

    all_coverage_rate = []
    all_rmse = []
    all_log_p = []
    for re_repeat in range(repeat_time):
        low = N * re_repeat
        high = N * (re_repeat + 1)
        X = all_X[low:high]
        X_sq = (X ** 2).sum(dim=1, keepdim=True)
        dist2 = X_sq + M_sq.unsqueeze(0) - 2 * (X @ M.t())

        min_dist2, argmin = dist2.min(dim=1)

        log_p = flow.compute_iid_log_density(X)

        coverage_count = torch.unique(argmin).numel()
        rmse = (min_dist2.mean()).sqrt().item()
        log_p = log_p.mean().item()

        all_coverage_rate.append(coverage_count / num_modes)
        all_rmse.append(rmse)
        all_log_p.append(log_p)
    marginal_coverage_summary = summarize_with_ci(all_coverage_rate)
    marginal_rmse_summary = summarize_with_ci(all_rmse)
    marginal_log_p_summary = summarize_with_ci(all_log_p)

    return coverage_summary, rmse_summary, log_p_summary, marginal_coverage_summary, marginal_rmse_summary, marginal_log_p_summary

In [ ]:
import json

base_seed = 0
repeat_time = 10000
f_0 = 1.0

setup_list = []
for force_name in ['dpp']:
    for s_proj in ['none', 'soft', 'hard']:
        setup = {
            'force_name': force_name,
            's_proj': s_proj,
            'f_0': f_0,
        }
        setup_list.append(setup)

results = []
for setup in tqdm(setup_list):
    force_name = setup['force_name']
    s_proj = setup['s_proj']
    f_0 = setup['f_0']

    coverage_summary, rmse_summary, log_p_summary, marginal_coverage_summary, marginal_rmse_summary, marginal_log_p_summary = evaluate_mode_coverage(
        flow, force_name, mus_torch, s_proj=s_proj, f_0=f_0,\
                base_seed=base_seed, repeat_time=repeat_time
    )
    results.append({
        'setup': setup,
        'coverage_summary': coverage_summary,
        'rmse_summary': rmse_summary,
        'log_p_summary': log_p_summary,
        'marginal_coverage_summary': marginal_coverage_summary,
        'marginal_rmse_summary': marginal_rmse_summary,
        'marginal_log_p_summary': marginal_log_p_summary,
        'repeat_time': repeat_time
    })

results.sort(key=lambda x: x['coverage_summary']['mean'], reverse=True)

with open('mode_coverage_all.json', 'w') as f:
    json.dump(results, f, indent=4)

In [ ]:
def evaluate_mode_coverage_iid(
    flow, mode_means, num_samples=10, repeat_time=100, base_seed=0.0
):
    device = mode_means.device
    num_modes = mode_means.size(0)

    M = mode_means.to(device)
    M_sq = (M ** 2).sum(dim=1)

    def _single_run(run_idx: int) -> int:
        torch.manual_seed(base_seed + run_idx)
        torch.cuda.manual_seed(base_seed + run_idx)
        random.seed(base_seed + run_idx)
        np.random.seed(base_seed + run_idx)

        out = flow.iid_sample(
            n_samples=num_samples
        )

        X = out["samples"].detach().to(device)
        X_sq = (X ** 2).sum(dim=1, keepdim=True)
        dist2 = X_sq + M_sq.unsqueeze(0) - 2 * (X @ M.t())

        min_dist2, argmin = dist2.min(dim=1)

        log_p = flow.compute_iid_log_density(X)
        return torch.unique(argmin).numel(), (min_dist2.mean()).sqrt().item(), log_p.mean().item()

    all_coverage_rate = []
    all_rmse = []
    all_log_p = []
    for i in range(repeat_time):
        coverage_count, rmse, log_p = _single_run(i)
        all_coverage_rate.append(coverage_count / num_modes)
        all_rmse.append(rmse)
        all_log_p.append(log_p)

    coverage_summary = summarize_with_ci(all_coverage_rate)
    rmse_summary = summarize_with_ci(all_rmse)
    log_p_summary = summarize_with_ci(all_log_p)
    return coverage_summary, rmse_summary, log_p_summary

base_seed = 0
repeat_time = 10000

coverage_summary, rmse_summary, log_p_summary = evaluate_mode_coverage_iid(flow, mus_torch, repeat_time=repeat_time, base_seed=base_seed)

print(coverage_summary, rmse_summary, log_p_summary)

# Code for density estimation

In [ ]:
import torch
import matplotlib.pyplot as plt
from tqdm import tqdm

torch.manual_seed(0)

N = 10

all_dimensions = 2
radius = 1.0
sigma_XY = 0.01
angles = torch.linspace(0, 2 * torch.pi, N + 1)[:-1]
mus_torch = torch.zeros((N, all_dimensions))
mus_torch[:, :2] = torch.stack([radius * torch.cos(angles), radius * torch.sin(angles)], dim=1)
mus_torch[:, 0] = mus_torch[:, 0] + 1.0
sigmas_torch = sigma_XY * torch.ones((N, all_dimensions))

weights_torch = torch.tensor([512 / (2**i) for i in range(N)], dtype=torch.float)
weights_torch /= weights_torch.sum()

from typing import Sequence

def expected_unique(weights: Sequence[float], n: int = 10) -> float:
    total = float(sum(weights))
    if total <= 0:
        raise ValueError("Sum of weights must be positive.")
    ps = [w / total for w in weights]
    return sum(1.0 - (1.0 - p) ** n for p in ps)
print('expected coverage:', expected_unique(weights_torch.tolist(), n=N) / N)

def sample_gmm_torch(n_samples):
    samples = []
    component_choices = torch.multinomial(weights_torch, n_samples, replacement=True)
    for k in component_choices:
        mean = mus_torch[k]
        std = torch.sqrt(sigmas_torch[k])
        sample = mean + std * torch.randn(all_dimensions)
        samples.append(sample)
    return torch.stack(samples)

gmm_samples_torch = sample_gmm_torch(10000)

plt.figure(figsize=(6,6))
plt.scatter(gmm_samples_torch[:,0].numpy(), gmm_samples_torch[:,1].numpy(), alpha=0.1, s=10, label="Samples")
plt.scatter(mus_torch[:,0].numpy(), mus_torch[:,1].numpy(), color='red', marker='x', s=100, label="Component Means")
plt.title('Samples from 2D Gaussian Mixture Model')
plt.xlabel('x')
plt.ylabel('y')
plt.axis('equal')
plt.grid(True)
plt.legend()
plt.show()

flow = FlowToGMM(mus_torch, sigmas_torch, weights_torch)

In [ ]:
output = flow.non_iid_sample('dpp_ho', 10, steps=100, save_path=True, f_0=1.0, s_proj='soft')
samples = output['samples']
path_history = output['path_history']
velocity_history = output['velocity_history']
force_history = output['force_history']

import matplotlib.pyplot as plt

arrow_length = 0.2

plt.figure(figsize=(10, 10))

for i in range(len(samples)):
    path = torch.stack([p[i] for p in path_history])
    plt.plot(path[:, 0].cpu(), path[:, 1].cpu(), alpha=0.3, linewidth=0.5)

subsample = 5
for step_idx in range(0, len(path_history) - 1, subsample):
    for i in range(len(samples)):
        pos = path_history[step_idx][i].cpu()

        if step_idx < len(velocity_history):
            v = velocity_history[step_idx][i].cpu()
            v_norm = v / (torch.norm(v) + 1e-8) * 0.3
            plt.arrow(pos[0], pos[1], v_norm[0] * arrow_length, v_norm[1] * arrow_length,
                     head_width=0.02, head_length=0.02, fc='blue', ec='blue', alpha=0.4)

        if step_idx < len(force_history):
            f = force_history[step_idx][i].cpu()
            f_norm = f / (torch.norm(f) + 1e-8) * 0.3
            plt.arrow(pos[0], pos[1], f_norm[0] * arrow_length, f_norm[1] * arrow_length,
                     head_width=0.02, head_length=0.02, fc='red', ec='red', alpha=0.4)

plt.scatter(samples[:, 0].cpu(), samples[:, 1].cpu(), s=50, c='green', marker='o', label='Final samples', zorder=5)
plt.scatter(mus_torch[:, 0].cpu(), mus_torch[:, 1].cpu(), s=100, c='black', marker='x', label='Means', zorder=5)

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color='green', marker='o', linestyle='', markersize=8, label='Final samples'),
    Line2D([0], [0], color='black', marker='x', linestyle='', markersize=8, label='Means'),
    Line2D([0], [0], color='blue', marker='>', linestyle='', markersize=8, label='Velocity direction'),
    Line2D([0], [0], color='red', marker='>', linestyle='', markersize=8, label='Force direction'),
    Line2D([0], [0], color='gray', linestyle='-', linewidth=1, label='Sample paths')
]

plt.axis('equal')
plt.grid(True, alpha=0.3)
plt.legend(handles=legend_elements)
plt.title('Flow paths with velocity and force directions')
plt.show()

In [ ]:
N = 10
train_data_size = 1000
data = []
train_seed = 1000000000
for seed in tqdm(range(train_data_size)):
    out = flow.non_iid_sample(
        'dpp_ho', N,
        s_proj='soft',
        f_0=1.0, save_path=True,
        seed=train_seed + seed
    )
    samples = out["samples"]
    data.append(samples)

In [ ]:
import torch
import torch.nn as nn
import math
import torch.nn.functional as F
from torch.nn.utils import spectral_norm as apply_sn
from torch.nn.utils import remove_spectral_norm

def remove_sn_(module: nn.Module):
    for m in module.modules():
        if isinstance(m, (nn.Linear)):
            try:
                remove_spectral_norm(m)
                print(f"Removed SN from {m}")
            except Exception:
                pass

class TimestepEmbedder(nn.Module):
    def __init__(self, dim, fdim=128):
        super().__init__()
        self.fdim = fdim
        lin1 = nn.Linear(fdim, dim)
        lin2 = nn.Linear(dim, dim)
        self.mlp = nn.Sequential(lin1, nn.SiLU(), lin2)

    def forward(self, t):
        half = self.fdim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / max(half, 1))
        freqs = freqs.to(t.dtype)
        if t.ndim == 2 and t.shape[1] == 1:
            t = t[:, 0]
        angles = t[:, None] * freqs[None]
        emb = torch.cat([torch.cos(angles), torch.sin(angles)], -1)
        if self.fdim % 2:
            emb = F.pad(emb, (0, 1))
        return self.mlp(emb)

class MLPBlock(nn.Module):
    def __init__(
        self, ci, co, tdim,
        use_sn: bool = True,
        n_power_iterations: int = 2,
        s_max: float = 1.0
    ):
        super().__init__()
        self.s_max = s_max

        fc1 = nn.Linear(ci, co)
        fc2 = nn.Linear(co, co)
        to_scale = nn.Linear(tdim, co * 2)
        to_shift = nn.Linear(tdim, co * 2)

        if use_sn:
            fc1 = apply_sn(fc1, n_power_iterations=n_power_iterations)
            fc2 = apply_sn(fc2, n_power_iterations=n_power_iterations)

        self.fc1 = fc1
        self.fc2 = fc2
        self.to_scale = to_scale
        self.to_shift = to_shift

        nn.init.zeros_(self.to_shift.weight)
        nn.init.zeros_(self.to_shift.bias)
        nn.init.zeros_(self.to_scale.weight)
        nn.init.zeros_(self.to_scale.bias)

    def forward(self, x, t_emb):
        scale_all = torch.tanh(self.to_scale(t_emb)) * self.s_max
        shift_all = self.to_shift(t_emb)
        s1, s2 = scale_all.chunk(2, dim=1)
        b1, b2 = shift_all.chunk(2, dim=1)

        h = self.fc1(x)
        h = h * (1.0 + s1) + b1
        h = F.silu(h)

        h = self.fc2(h)
        h = h * (1.0 + s2) + b2
        h = F.silu(h)

        return h

class DVNet(nn.Module):
    def __init__(
        self,
        c, ch=64,
        use_sn: bool = True,
        n_power_iterations: int = 2,
        s_max: float = 1.0,
        depth: int = 2,
    ):
        super().__init__()
        tdim = ch * 4

        self.temb = TimestepEmbedder(
            tdim
        )

        blocks = []
        blocks.append(
            MLPBlock(
                ci=c, co=ch, tdim=tdim,
                use_sn=use_sn,
                n_power_iterations=n_power_iterations,
                s_max=s_max,
            )
        )
        for _ in range(depth - 1):
            blocks.append(
                MLPBlock(
                    ci=ch, co=ch, tdim=tdim,
                    use_sn=use_sn,
                    n_power_iterations=n_power_iterations,
                    s_max=s_max,
                )
            )
        self.blocks = nn.ModuleList(blocks)
        self.out = nn.Linear(ch, c)

    def forward(self, x, t):
        if t.ndim == 2 and t.shape[1] == 1:
            t = t[:, 0]
        t_emb = self.temb(t)
        h = x
        for block in self.blocks:
            h = block(h, t_emb)
        return self.out(h)

dvnet = DVNet(c=all_dimensions, use_sn=False)
dvnet.train()

batch_size = 1000

class EfficientXLoader:
    def __init__(self, data, batch_size):
        self.batch_size = batch_size
        self.x_matrix = torch.cat(data, dim=0)
        indexes = torch.randperm(self.x_matrix.shape[0])
        self.x_matrix = self.x_matrix[indexes]
        self.batch_num = self.x_matrix.shape[0] // batch_size

    def get_batch(self, batch_idx):
        start_idx = batch_idx * self.batch_size
        end_idx = start_idx + self.batch_size
        return self.x_matrix[start_idx:end_idx]

    def get_batch_num(self):
        return self.batch_num

gpu_x_loader = EfficientXLoader(data, batch_size)
optimizer = torch.optim.AdamW(dvnet.parameters(), lr=1e-4, weight_decay=1e-3)

epoch_loss_list = []
for epoch in range(100):
    dvnet.train()
    loss_list = []

    batch_num = gpu_x_loader.get_batch_num()
    for batch_idx in range(batch_num):
        X1 = gpu_x_loader.get_batch(batch_idx)
        X0 = torch.randn_like(X1)
        t = torch.rand(X1.shape[0], 1)
        Xt = X0 + t * (X1 - X0)
        Vt = flow.velocity_batched(Xt, t)
        target = X1 - X0 - Vt

        pred = dvnet(Xt, t)
        loss = F.mse_loss(pred, target)
        optimizer.zero_grad()
        total_loss = loss
        total_loss.backward()
        optimizer.step()
        loss_list.append(loss.item())
    epoch_loss_list.append(sum(loss_list) / len(loss_list))
dvnet.eval()


remove_sn_(dvnet)


import matplotlib.pyplot as plt
import numpy as np

window_size = 10
epoch_loss_array = np.array(epoch_loss_list)

num_windows = len(epoch_loss_array) // window_size
windowed_losses = []
window_epochs = []

for i in range(num_windows):
    start_idx = i * window_size
    end_idx = start_idx + window_size
    windowed_losses.append(epoch_loss_array[start_idx:end_idx].mean())
    window_epochs.append((start_idx + end_idx) / 2)

plt.figure(figsize=(10, 6))
plt.plot(window_epochs, windowed_losses, marker='o', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss (Window Average)')
plt.title(f'Training Loss (Window Size = {window_size})')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
del data, gpu_x_loader, optimizer

In [ ]:
from tqdm import tqdm

N = 10
test_data_size = 10000
test_data = []
test_seed = 1000000
for test_i in tqdm(range(test_data_size)):
    out = flow.non_iid_sample(
        'dpp_ho', N,
        s_proj='soft',
        f_0=1.0, save_path=True,
        seed=test_seed + test_i
    )
    samples = out["samples"]
    forces = out["force_history"]
    trajectories = out["path_history"]
    times = out["time_history"]

    test_data.append({
        "samples": samples,
        "forces": forces,
        "trajectories": trajectories,
        "times": times
    })

In [ ]:
test_results = []
for cur_data in tqdm(test_data):
    samples = cur_data["samples"]
    forces = cur_data["forces"]
    trajectories = cur_data["trajectories"]
    times = cur_data["times"]

    log_w = flow.importance_weight_computation(trajectories, forces, times, dvnet)
    log_w_ours = log_w.detach().clone()
    del log_w

    logw_fixed = flow.importance_weight_computation_fixed(samples, times, dvnet)
    log_w_ours_fixed = logw_fixed.detach().clone()
    del logw_fixed

    test_results.append({
        "log_w_ours": log_w_ours,
        "log_w_ours_fixed": log_w_ours_fixed,
        "samples": samples
    })

# Estimation

In [ ]:
from scipy import stats
def summarize_with_ci(values, confidence=0.95, round_to=6):
    arr = np.array(values)
    mean = np.mean(arr)
    std = np.std(arr, ddof=1)
    n = len(arr)
    se = std / np.sqrt(n)
    t_val = stats.t.ppf((1 + confidence) / 2, df=n - 1)
    ci_half = t_val * se

    mean = round(float(mean), round_to)
    std = round(float(std), round_to)
    ci_half = round(float(ci_half), round_to)
    return {
        "mean": mean,
        "std": std,
        "ci_half": ci_half
    }

def js_divergence(p, q, eps=1e-12):
    p = p / p.sum()
    q = q / q.sum()
    m = 0.5 * (p + q)
    kl_pm = (p * (torch.log(p + eps) - torch.log(m + eps))).sum()
    kl_qm = (q * (torch.log(q + eps) - torch.log(m + eps))).sum()
    return 0.5 * (kl_pm + kl_qm)

def sample_classification(samples):
    X = samples
    X_sq = (X ** 2).sum(dim=1, keepdim=True)

    M = mus_torch.to(X.device)
    M_sq = (M ** 2).sum(dim=1)

    dist2 = X_sq + M_sq.unsqueeze(0) - 2 * (X @ M.t())

    _, argmin = dist2.min(dim=1)

    argmin_one_hot = torch.nn.functional.one_hot(argmin, num_classes=N).float()
    return argmin_one_hot

In [ ]:
gt_iid_data_size = 1000000
gt_iid_seed = 2000000
out = flow.iid_sample(
    gt_iid_data_size, seed=gt_iid_seed
)
iid_one_hot = sample_classification(out["samples"])
gt_prob = iid_one_hot.mean(dim=0)
print(gt_prob)
del out

In [ ]:
iid_results = []
for test_i in range(test_data_size):
    out = flow.iid_sample(
        N, seed=test_i
    )
    iid_one_hot = sample_classification(out["samples"])
    iid_prediction = iid_one_hot.mean(dim=0)
    js_value = js_divergence(gt_prob, iid_prediction)
    iid_results.append(js_value)
    del out

iid_js_summary = summarize_with_ci(iid_results)
print(iid_js_summary)

In [ ]:
all_results = {
    'equal_weight': [],
    'ours': [],
    'ours_fixed': [],
}

for cur_test_result in test_results:
    X = cur_test_result["samples"]
    X_sq = (X ** 2).sum(dim=1, keepdim=True)

    M = mus_torch.to(X.device)
    M_sq = (M ** 2).sum(dim=1)

    dist2 = X_sq + M_sq.unsqueeze(0) - 2 * (X @ M.t())

    _, argmin = dist2.min(dim=1)

    argmin_one_hot = torch.nn.functional.one_hot(argmin, num_classes=N).float()

    naive_mean = argmin_one_hot.mean(dim=0)
    all_results['equal_weight'].append(js_divergence(gt_prob, naive_mean))

    log_w_ours = cur_test_result["log_w_ours"]
    w_ours_normalized = F.softmax(log_w_ours.view(-1)).view(-1, 1)
    ours_mean_normalized = (w_ours_normalized * argmin_one_hot).sum(dim=0)
    all_results['ours'].append(js_divergence(gt_prob, ours_mean_normalized))

    log_w_ours_fixed = cur_test_result["log_w_ours_fixed"]
    w_ours_fixed_normalized = F.softmax(log_w_ours_fixed.view(-1)).view(-1, 1)
    ours_mean_fixed_normalized = (w_ours_fixed_normalized * argmin_one_hot).sum(dim=0)
    all_results['ours_fixed'].append(js_divergence(gt_prob, ours_mean_fixed_normalized))

all_summary_results = {key: summarize_with_ci(value) for key, value in all_results.items()}

In [ ]:
all_summary_results

In [ ]:
final_results = []

group_size = N
all_samples = torch.cat([test_result["samples"] for test_result in test_results], dim=0)
knn_estimation = flow.compute_non_iid_log_density_knn(all_samples)
all_observed_log_density = knn_estimation[:,0]
all_observed_confidence = knn_estimation[:,1]
all_observed_confidence = all_observed_confidence.view(-1, N).min(dim=1).values
percentile_50 = torch.quantile(all_observed_confidence, 0.5)
confident_masks = all_observed_confidence >= percentile_50
print('keep rate', confident_masks.float().mean())
confident_masks = confident_masks.tolist()

all_observed_log_density_groups = []
start_idx = 0
for _ in range(len(test_results)):
    all_observed_log_density_groups.append(all_observed_log_density[start_idx:start_idx + group_size])
    start_idx += group_size

filtered_all_samples = []
for test_result, observed_log_density, confident_mask in zip(test_results, all_observed_log_density_groups, confident_masks):
    if confident_mask == 0:
        continue
    samples = test_result["samples"]
    iid_log_density = flow.compute_iid_log_density(samples)
    gt_log_w = iid_log_density - observed_log_density
    final_results.append({
        "gt": gt_log_w,
        "pred_ours": test_result["log_w_ours"],
        "pred_ours_fixed": test_result["log_w_ours_fixed"],
        "samples": samples
    })
    filtered_all_samples.append(samples)
filtered_all_samples = torch.cat(filtered_all_samples, dim=0)

In [ ]:
import math
from typing import List, Dict
import numpy as np
import torch

def rank_descending(values: np.ndarray) -> np.ndarray:
    n = values.size
    order = np.argsort(-values, kind="mergesort")
    ranks = np.empty(n, dtype=float)
    i = 0
    while i < n:
        j = i
        while j + 1 < n and values[order[j + 1]] == values[order[i]]:
            j += 1
        ranks[order[i:j + 1]] = (i + j) / 2 + 1
        i = j + 1
    return ranks


def kendall_tau_b(pred_scores: np.ndarray, gt_scores: np.ndarray) -> float:
    n = len(pred_scores)
    concordant = discordant = ties_pred = ties_gt = 0
    for i in range(n):
        for j in range(i + 1, n):
            dp = np.sign(pred_scores[i] - pred_scores[j])
            dg = np.sign(gt_scores[i] - gt_scores[j])
            if dp == 0 and dg == 0:
                continue
            if dp == 0:
                ties_pred += 1
            elif dg == 0:
                ties_gt += 1
            elif dp == dg:
                concordant += 1
            else:
                discordant += 1
    denom = math.sqrt((concordant + discordant + ties_pred) *
                      (concordant + discordant + ties_gt))
    return 0.0 if denom == 0 else (concordant - discordant) / denom


def spearman_rho(pred_scores: np.ndarray, gt_scores: np.ndarray) -> float:
    r_pred = rank_descending(pred_scores)
    r_gt = rank_descending(gt_scores)
    return float(np.corrcoef(r_pred, r_gt)[0, 1])


def _normalize_graded_relevance_to_unit(rel: np.ndarray) -> np.ndarray:
    rel = rel.astype(float)
    if rel.min() < 0:
        rel = rel - rel.min()
    mx = rel.max()
    if mx == 0.0:
        return rel
    if mx > 1.0:
        rel = rel / mx
    return rel


def soft_average_precision_gAP(gt_scores: np.ndarray, pred_scores: np.ndarray) -> float:
    order = np.argsort(-pred_scores, kind="mergesort")
    rel_sorted = gt_scores[order]
    rel_sorted = _normalize_graded_relevance_to_unit(rel_sorted)

    total_gain = rel_sorted.sum()
    if total_gain == 0.0:
        return 0.0

    precision_prefix = np.cumsum(rel_sorted) / (np.arange(len(rel_sorted)) + 1)
    weights = rel_sorted / total_gain
    sap = float(np.sum(precision_prefix * weights))

    if sap < 0.0:
        sap = 0.0
    if sap > 1.0 and sap - 1.0 < 1e-12:
        sap = 1.0
    return sap


def evaluate_single_pair(
    gt_scores_tensor: torch.Tensor,
    pred_scores_tensor: torch.Tensor,
) -> Dict[str, float]:
    assert gt_scores_tensor.ndim == pred_scores_tensor.ndim == 1, "Inputs must be 1D."
    assert gt_scores_tensor.numel() == pred_scores_tensor.numel() == 10, "Expected length = 10."

    gt = gt_scores_tensor.detach().cpu().numpy().astype(float)
    pred = pred_scores_tensor.detach().cpu().numpy().astype(float)

    tau_b_value = kendall_tau_b(pred, gt)
    rho_value = spearman_rho(pred, gt)
    soft_ap_value = soft_average_precision_gAP(gt, pred)

    if np.all((gt == 0) | (gt == 1)):
        order = np.argsort(-pred, kind="mergesort")
        y = gt[order]
        if y.sum() > 0:
            prec = np.cumsum(y) / (np.arange(len(y)) + 1)
            ap_bin = float((prec * y).sum() / y.sum())
            assert abs(ap_bin - soft_ap_value) < 1e-8, "Binary AP != gAP (should coincide)."

    assert -1.0 <= tau_b_value <= 1.0, "Kendall tau-b out of bounds."
    assert -1.0 <= rho_value <= 1.0, "Spearman rho out of bounds."
    assert 0.0 - 1e-12 <= soft_ap_value <= 1.0 + 1e-12, "graded AP out of [0,1]."

    return {
        "kendall_tau_b": float(tau_b_value),
        "spearman_rho": float(rho_value),
        "soft_AP": float(soft_ap_value),
    }


def evaluate_batch(records: List[Dict[str, torch.Tensor]]) -> Dict[str, float]:
    tau_vals, rho_vals, gap_vals = [], [], []

    for rec in records:
        res = evaluate_single_pair(rec["gt"], rec["pred"])
        tau_vals.append(res["kendall_tau_b"])
        rho_vals.append(res["spearman_rho"])
        gap_vals.append(res["soft_AP"])

    return {
        "kendall_tau_b": summarize_with_ci(tau_vals),
        "spearman_rho": summarize_with_ci(rho_vals),
        "soft_AP": summarize_with_ci(gap_vals),
    }

In [ ]:
import numpy as np

from scipy import stats
def summarize_with_ci(values, confidence=0.95, round_to=6):
    arr = np.array(values)
    mean = np.mean(arr)
    std = np.std(arr, ddof=1)
    n = len(arr)
    se = std / np.sqrt(n)
    t_val = stats.t.ppf((1 + confidence) / 2, df=n - 1)
    ci_half = t_val * se

    mean = round(float(mean), round_to)
    std = round(float(std), round_to)
    ci_half = round(float(ci_half), round_to)
    return {
        "mean": mean,
        "std": std,
        "ci_half": ci_half
    }

final_results_ours = [{"gt": result["gt"], "pred": result["pred_ours"]} for result in final_results]
eval_scores = evaluate_batch(final_results_ours)
print('ours')
print(eval_scores)

final_results_ours_fixed = [{"gt": result["gt"], "pred": result["pred_ours_fixed"]} for result in final_results]
eval_scores = evaluate_batch(final_results_ours_fixed)
print('ours_fixed')
print(eval_scores)

In [ ]:
def compute_log_w_mse(log_w_pred, log_w_gt):
    return ((log_w_pred - log_w_gt) ** 2).mean()

def compute_bias(log_w_pred, log_w_gt):
    return (log_w_pred - log_w_gt).mean()


mse_list = []
bias_list = []
for result in final_results:
    pred_ours = result["pred_ours"]
    gt = result["gt"]
    mse = compute_log_w_mse(pred_ours, gt)
    bias = compute_bias(pred_ours, gt)
    mse_list.append(mse)
    bias_list.append(bias)
mse_summary = summarize_with_ci(mse_list)
bias_summary = summarize_with_ci(bias_list)
print(f'ours, mse = {mse_summary}, bias = {bias_summary}')

mse_list = []
bias_list = []
for result in final_results:
    pred_ours = result["pred_ours_fixed"]
    gt = result["gt"]
    mse = compute_log_w_mse(pred_ours, gt)
    bias = compute_bias(pred_ours, gt)
    mse_list.append(mse)
    bias_list.append(bias)
mse_summary = summarize_with_ci(mse_list)
bias_summary = summarize_with_ci(bias_list)
print(f'ours_fixed, mse = {mse_summary}, bias = {bias_summary}')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
import torch

def plot_log_density_smooth_fast(
    x,
    log_density,
    gridsize=400,
    sigma=2.0,
    padding=0.02,
    vmin=None,
    vmax=None,
    cmap="viridis",
    percentile_clip=(2, 98),
    show_colorbar=True,
    return_artists=False,
    title="Log-density (smoothed)",
    xlabel="X",
    ylabel="Y",
    *,
    fill_blank=True,
    blank_margin=2.0,
    zmin=None,
    zmax=None,
    xy_extent=None,
    floating_label='log density',
):
    try:
        X = x[:, 0].detach().cpu().numpy()
        Y = x[:, 1].detach().cpu().numpy()
        Z = log_density.detach().cpu().numpy()
    except AttributeError:
        X = np.asarray(x)[:, 0]
        Y = np.asarray(x)[:, 1]
        Z = np.asarray(log_density)

    mask = np.ones_like(Z, dtype=bool)
    if zmin is not None:
        mask &= (Z >= zmin)
    if zmax is not None:
        mask &= (Z <= zmax)
    if not np.any(mask):
        raise ValueError("No data left after zmin/zmax filtering.")
    X, Y, Z = X[mask], Y[mask], Z[mask]

    if xy_extent is None:
        xmin, xmax = np.min(X), np.max(X)
        ymin, ymax = np.min(Y), np.max(Y)
        rx, ry = xmax - xmin, ymax - ymin
        xmin -= padding * rx; xmax += padding * rx
        ymin -= padding * ry; ymax += padding * ry
    else:
        xmin, xmax, ymin, ymax = xy_extent

    if isinstance(gridsize, int):
        nx = ny = gridsize
    else:
        nx, ny = gridsize

    xedges = np.linspace(xmin, xmax, nx + 1)
    yedges = np.linspace(ymin, ymax, ny + 1)

    sum_w, _, _ = np.histogram2d(X, Y, bins=[xedges, yedges], weights=Z)
    cnt,   _, _ = np.histogram2d(X, Y, bins=[xedges, yedges])

    if sigma and sigma > 0:
        sum_w_s = gaussian_filter(sum_w, sigma=sigma, mode="nearest")
        cnt_s   = gaussian_filter(cnt,   sigma=sigma, mode="nearest")
    else:
        sum_w_s, cnt_s = sum_w, cnt

    with np.errstate(invalid="ignore", divide="ignore"):
        grid = np.divide(sum_w_s, cnt_s, out=np.full_like(sum_w_s, np.nan), where=cnt_s > 0)

    if fill_blank:
        if np.isfinite(grid).any():
            min_log = float(np.nanmin(grid[np.isfinite(grid)]))
        else:
            min_log = -20.0
        grid[~np.isfinite(grid)] = min_log - float(blank_margin)

    if vmin is None or vmax is None:
        finite_vals = grid[np.isfinite(grid)]
        if finite_vals.size > 0:
            lo, hi = np.percentile(finite_vals, percentile_clip)
            if vmin is None: vmin = float(lo)
            if vmax is None: vmax = float(hi)
            if vmin >= vmax:
                vmin, vmax = float(finite_vals.min()), float(finite_vals.max())
        else:
            vmin, vmax = -1.0, 1.0

    extent = (xedges[0], xedges[-1], yedges[0], yedges[-1])
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(
        grid.T,
        extent=extent,
        origin="lower",
        aspect="auto",
        vmin=vmin,
        vmax=vmax,
        cmap=cmap,
    )
    if show_colorbar:
        fig.colorbar(im, ax=ax, label=floating_label)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    fig.tight_layout()

    if return_artists:
        return fig, ax, im, extent, grid
    return None


def compute_common_scale_and_extent(
    xs_list, zs_list,
    *,
    zmin=None, zmax=None,
    percentile_clip=(2, 98),
    padding=0.02
):
    X_all, Y_all = [], []
    for x in xs_list:
        try:
            x_np = x.detach().cpu().numpy()
        except AttributeError:
            x_np = np.asarray(x)
        X_all.append(x_np[:, 0])
        Y_all.append(x_np[:, 1])
    X_all = np.concatenate(X_all)
    Y_all = np.concatenate(Y_all)

    xmin, xmax = float(np.min(X_all)), float(np.max(X_all))
    ymin, ymax = float(np.min(Y_all)), float(np.max(Y_all))
    rx, ry = xmax - xmin, ymax - ymin
    xy_extent = (xmin - padding*rx, xmax + padding*rx,
                 ymin - padding*ry, ymax + padding*ry)

    vals = []
    for z in zs_list:
        try:
            z_np = z.detach().cpu().numpy()
        except AttributeError:
            z_np = np.asarray(z)
        zf = z_np[np.isfinite(z_np)]
        if zmin is not None:
            zf = zf[zf >= zmin]
        if zmax is not None:
            zf = zf[zf <= zmax]
        if zf.size:
            vals.append(zf.ravel())

    if not vals:
        raise ValueError("No finite z values left after filtering to compute global color scale.")

    stacked = np.concatenate(vals)
    vmin_global = float(np.percentile(stacked, percentile_clip[0]))
    vmax_global = float(np.percentile(stacked, percentile_clip[1]))
    if vmin_global >= vmax_global:
        vmin_global, vmax_global = float(stacked.min()), float(stacked.max())

    return vmin_global, vmax_global, xy_extent


z_pred_all = torch.cat([res["pred_ours"] for res in final_results], dim=0)
z_gt_all   = torch.cat([res["gt"]        for res in final_results], dim=0)

zmin_bound, zmax_bound = -100.0, 100.0
percentile_clip = (2, 98)
padding = 0.02

zs_list = [z_pred_all, z_gt_all]
xs_list = [filtered_all_samples] * len(zs_list)

vmin_g, vmax_g, xy_extent = compute_common_scale_and_extent(
    xs_list, zs_list,
    zmin=zmin_bound, zmax=zmax_bound,
    percentile_clip=percentile_clip,
    padding=padding
)

plot_log_density_smooth_fast(
    filtered_all_samples, z_pred_all,
    gridsize=1024,
    sigma=1,
    cmap="viridis",
    fill_blank=False,
    blank_margin=0.0,
    zmin=zmin_bound,
    zmax=zmax_bound,
    vmin=vmin_g,
    vmax=vmax_g,
    xy_extent=xy_extent,
    title="pred log w",
    floating_label='pred log w',
)

plot_log_density_smooth_fast(
    filtered_all_samples, z_gt_all,
    gridsize=1024,
    sigma=1,
    cmap="viridis",
    fill_blank=False,
    blank_margin=0.0,
    zmin=zmin_bound,
    zmax=zmax_bound,
    vmin=vmin_g,
    vmax=vmax_g,
    xy_extent=xy_extent,
    title="gt log w",
    floating_label='gt log w',
)

absdiff = (z_pred_all - z_gt_all).abs()

vmin_err, vmax_err, xy_extent_err = compute_common_scale_and_extent(
    [filtered_all_samples], [absdiff],
    zmin=0.0,
    zmax=None,
    percentile_clip=(2, 98),
    padding=0.02
)
vmin_err = 0.0

plot_log_density_smooth_fast(
    filtered_all_samples, absdiff,
    gridsize=1024,
    sigma=1,
    cmap="viridis",
    fill_blank=False,
    blank_margin=0.0,
    zmin=0.0,
    zmax=None,
    vmin=vmin_err,
    vmax=vmax_err,
    xy_extent=xy_extent_err,
    title="|pred log w - gt log w|",
    floating_label='|pred log w - gt log w|',
)